In [12]:
from dataclasses import dataclass
import os
from openai import OpenAI

@dataclass(frozen=True)
class Provider:
    """ One provider to relaiably route all requests around inference providers"""
    name:str
    env_var:str
    is_free:bool
    base_url:str|None
    model:str

PROVIDERS = [
    Provider("OpenAI","OPENAI_API_KEY",False,None,"gpt-4o-mini"),
    Provider("Groq", "GROQ_API_KEY", True, "https://api.groq.com/openai/v1", "openai/gpt-oss-120b"),
]

def select_provider()->Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider
    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set, add one of {expected} to your environment variables")

def build_client(provider:Provider)->OpenAI:
    api_key = os.getenv(provider.env_var)
    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(api_key=api_key,model=provider.model)

def have_any_key()->bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")

Found a provider key.


In [ ]:
def llm_reply(prompt:str)->str:
    provider = select_provider()
    print(f"Using {provider.name} provider")
    client = build_client(provider)
    result = client.chat.completions.create(
        model=provider.model,
        messages=[
            {"role":"user",
             "content":prompt}
        ]
    )
    return result.choices[0].message.content

In [14]:
prompt="Who was the prime minister of UK in 2020? Anser in single sentence"
try:
    print(llm_reply(prompt))
except Exception as e:
    print(f"Something went wrong {e}")

Using OpenAI provider
The Prime Minister of the UK in 2020 was Boris Johnson.


In [11]:
prompt="Who was before him prime minister?"
try:
    print(llm_reply(prompt))
except Exception as e:
    print(f"Something went wrong {e}")

Using OpenAI provider
To provide an accurate answer, could you please specify which prime minister you are referring to?


In [15]:
def chat_reply(messages:list[dict])->str:
    provider = select_provider()
    client = build_client(provider)
    response =  client.chat.completions.create(
        model=provider.model,
        max_tokens=1000,
        messages=messages
    ) 
    return response.choices[0].message.content

In [24]:
converstaion=[]

In [25]:
converstaion.append({
    "role":"user",
    "content":"Who was the prime minister of UK in 2020?"
    })
print(converstaion)

[{'role': 'user', 'content': 'Who was the prime minister of UK in 2020?'}]


In [26]:
reply_from_llm=chat_reply(converstaion)
print(reply_from_llm)
converstaion.append({
    "role":"assistant",
    "content":reply_from_llm
})

In 2020, the Prime Minister of the United Kingdom was Boris Johnson. He became Prime Minister on July 24, 2019, and served in that role throughout 2020.


In [27]:
converstaion.append({
     "role":"user",
     "content":"Who was before that?"
})

In [28]:
converstaion

[{'role': 'user', 'content': 'Who was the prime minister of UK in 2020?'},
 {'role': 'assistant',
  'content': 'In 2020, the Prime Minister of the United Kingdom was Boris Johnson. He became Prime Minister on July 24, 2019, and served in that role throughout 2020.'},
 {'role': 'user', 'content': 'Who was before that?'}]

In [29]:
reply_from_llm=chat_reply(converstaion)
print(reply_from_llm)
converstaion.append({
    "role":"assistant",
    "content":reply_from_llm
})

Before Boris Johnson, the Prime Minister of the United Kingdom was Theresa May. She served as Prime Minister from July 13, 2016, until July 24, 2019.


In [30]:
converstaion

[{'role': 'user', 'content': 'Who was the prime minister of UK in 2020?'},
 {'role': 'assistant',
  'content': 'In 2020, the Prime Minister of the United Kingdom was Boris Johnson. He became Prime Minister on July 24, 2019, and served in that role throughout 2020.'},
 {'role': 'user', 'content': 'Who was before that?'},
 {'role': 'assistant',
  'content': 'Before Boris Johnson, the Prime Minister of the United Kingdom was Theresa May. She served as Prime Minister from July 13, 2016, until July 24, 2019.'}]